In [1]:
import zipfile
import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import optuna # <-- NUEVO: Importar optuna

from ray import tune
from ray.tune.schedulers import ASHAScheduler
from ray.tune.search.optuna import OptunaSearch

2025-06-16 15:10:40.203893: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750101040.217646   56794 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750101040.221256   56794 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750101040.234668   56794 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750101040.234686   56794 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750101040.234687   56794 computation_placer.cc:177] computation placer alr

In [2]:
zip_file_path = '../data/MeIA2025-Reto-01.zip'

extracted_folder_path = '../data/extracted_corpus/'

# 2. Crear la carpeta de extracción si no existe
if not os.path.exists(extracted_folder_path):
    os.makedirs(extracted_folder_path)
    print(f"Carpeta '{extracted_folder_path}' creada.")
# 3. Descomprimir archivo  
try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extracted_folder_path)
    print(f"'{zip_file_path}' descomprimido exitosamente en '{extracted_folder_path}'.")
except FileNotFoundError:
    print(f"Error: El archivo ZIP no se encontró en '{zip_file_path}'. Verifica la ruta.")
except Exception as e:
    print(f"Ocurrió un error al descomprimir el archivo: {e}")
    
# 4. Listar los archivos descomprimidos (para verificar)
print("\nArchivos en la carpeta del corpus:")
corpus_files = os.listdir(extracted_folder_path)
for file_name in corpus_files:
    print(f"- {file_name}")


'../data/MeIA2025-Reto-01.zip' descomprimido exitosamente en '../data/extracted_corpus/'.

Archivos en la carpeta del corpus:
- Datos-MeIA-Reto-01


In [3]:
# Definir la ruta base donde se extrajo el contenido del ZIP
# Asegúrate de que esta ruta sea correcta relativa a tu notebook test.ipynb
# Si tu notebook está en 'notebooks/' y la extracción está en 'data/extracted_corpus/Datos-MelA-Reto-01/'
base_extracted_path = '../data/extracted_corpus/Datos-MeIA-Reto-01/'

# Rutas completas a los archivos XLSX
train_file_path = os.path.join(base_extracted_path, 'MeIA_2025_train.xlsx')
test_file_path = os.path.join(base_extracted_path, 'MeIA_2025_test_wo_labels.xlsx')

print(f"Intentando cargar el archivo de entrenamiento desde: {train_file_path}")
print(f"Intentando cargar el archivo de prueba desde: {test_file_path}")

try:
    # Cargar el dataset de entrenamiento
    
    df_train = pd.read_excel(train_file_path)
    print(f"Datos de entrenamiento cargados correctamente")
    # Cargar el dataset de prueba (sin etiquetas)
    df_test = pd.read_excel(test_file_path)
    print(f"Datos de test cargados correctamente")


except FileNotFoundError:
    print(f"Error: Uno de los archivos XLSX no se encontró.")
    print(f"Asegúrate de que las rutas sean correctas: '{train_file_path}' y '{test_file_path}'")
    print(f"Y que la carpeta 'Datos-MelA-Reto-01' esté dentro de 'extracted_corpus'.")
except Exception as e:
    print(f"Ocurrió un error al cargar los archivos Excel: {e}")


Intentando cargar el archivo de entrenamiento desde: ../data/extracted_corpus/Datos-MeIA-Reto-01/MeIA_2025_train.xlsx
Intentando cargar el archivo de prueba desde: ../data/extracted_corpus/Datos-MeIA-Reto-01/MeIA_2025_test_wo_labels.xlsx
Datos de entrenamiento cargados correctamente
Datos de test cargados correctamente


### Analisis exploratorio

In [ ]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Review    5000 non-null   object 
 1   Polarity  5000 non-null   float64
 2   Town      5000 non-null   object 
 3   Region    5000 non-null   object 
 4   Type      5000 non-null   object 
dtypes: float64(1), object(4)
memory usage: 195.4+ KB


In [ ]:
df_train.head(5)

,Review,Polarity,Town,Region,Type
0,Un Restaurante te invita por su ambiente tan a...,2.0,Tlaquepaque,Jalisco,Restaurant
1,Pagamos 25 pesos por la entrada y no es gran c...,3.0,Bacalar,QuintanaRoo,Attractive
2,Mi esposa y yo nos alojamos en el Dreams por 4...,3.0,Tulum,QuintanaRoo,Hotel
3,"La única decepción puede no ser José Cuervo, p...",2.0,Tequila,Jalisco,Attractive
4,Cuando leí los comentarios sobre cómo son las ...,1.0,Isla_Mujeres,QuintanaRoo,Hotel


In [ ]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ID      2500 non-null   int64 
 1   Review  2500 non-null   object
 2   Town    2500 non-null   object
 3   Region  2500 non-null   object
 4   Type    2500 non-null   object
dtypes: int64(1), object(4)
memory usage: 97.8+ KB


In [ ]:
df_test.head(5)

,ID,Review,Town,Region,Type
0,0,Nos alojamos 5 noches en el Coco Beach Bungalo...,Tulum,QuintanaRoo,Hotel
1,1,"Es lindo, pero me hubiera gustado que nos come...",Tulum,QuintanaRoo,Attractive
2,2,"El lugar es muy bonito, lo tienen MUY descuida...",Tepoztlan,Morelos,Hotel
3,3,"Además de que Taxco es mágico, el zócalo tiene...",Taxco,Guerrero,Attractive
4,4,"Sin duda un lugar Mágico para visitar, debes i...",Tepoztlan,Morelos,Attractive


In [ ]:
#Polaridad
df_train['Polarity'].value_counts()

Polarity
5.0    1200
4.0    1100
3.0    1000
2.0     900
1.0     800
Name: count, dtype: int64

### Test

In [4]:
df=df_train.copy()

df = df.rename(columns={'Polarity': 'labels'})
df['labels'] = df['labels'] - 1
df['labels'] = df['labels'].astype(int)

In [5]:

# --- División de datos (tu código original) ---
df_train_split, df_eval_split = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df['labels']
)

# --- Tokenización (tu código original) ---
model_checkpoint = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

hf_train_dataset = Dataset.from_pandas(df_train_split[['Review', 'labels']])
hf_eval_dataset = Dataset.from_pandas(df_eval_split[['Review', 'labels']])

def tokenize_function(examples):
    return tokenizer(examples["Review"], truncation=True, padding="max_length", max_length=128)

tokenized_train_dataset = hf_train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset  = hf_eval_dataset.map(tokenize_function, batched=True)

# --- Métrica de Evaluación (tu código original) ---
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    # IMPORTANTE: La métrica del reto es 'weighted' F1
    f1 = f1_score(labels, predictions, average="weighted") 
    return {"f1": f1}

# --- NUEVO: Función para inicializar el modelo ---
# Optuna necesita una función que devuelva un modelo nuevo en cada trial
# para asegurar que no haya "fugas" de pesos entre entrenamientos.
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=5)

# --- MODIFICADO: Argumentos de Entrenamiento ---
# Estos son los argumentos base. Los que se van a tunear los definiremos en el espacio de búsqueda.
training_args = TrainingArguments(
    output_dir="./results_optuna",
    logging_dir="./logs_optuna",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    # La métrica ahora debe coincidir con la que devuelve compute_metrics
    metric_for_best_model="f1", 
    greater_is_better=True,
    fp16=True if torch.cuda.is_available() else False,
)

# --- MODIFICADO: Inicialización del Trainer ---
# Ya no pasamos un modelo, sino la función que lo inicializa.
trainer = Trainer(
    model_init=model_init, # <-- Usamos model_init en lugar de model=...
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# --- NUEVO: Definir el espacio de búsqueda de hiperparámetros para Optuna ---
def hp_space_optuna(trial: optuna.Trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-6, 5e-5, log=True),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 1, 4),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32]),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.3),
        "warmup_steps": trial.suggest_int("warmup_steps", 0, 1000),
    }

# --- NUEVO Y FINAL: Ejecutar la búsqueda de hiperparámetros ---
print("\nIniciando la búsqueda de hiperparámetros con Optuna...")
best_run = trainer.hyperparameter_search(
    direction="maximize",      # Queremos maximizar el F1-score
    backend="optuna",          # Usamos el backend de Optuna
    hp_space=hp_space_optuna,  # La función que define qué tunear
    n_trials=5,               # Número de combinaciones a probar (puedes aumentarlo)
    
)

print("\n¡Búsqueda completada!")
print(f"Mejor trial encontrado:")
print(f"  -> Valor de la métrica (f1): {best_run.objective}")
print(f"  -> Mejores Hiperparámetros: {best_run.hyperparameters}")

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

/tmp/ipykernel_56794/1415144865.py:53: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
[I 2025-06-16 15:11:07,765] A new study created in memory with name: no-name-5fef0423-c09f-460b-8859-0bb18384d423



Iniciando la búsqueda de hiperparámetros con Optuna...


Epoch,Training Loss,Validation Loss,F1
1,1.048000,1.015729,0.527213
2,1.050300,1.101200,0.471711
3,0.889700,1.087797,0.527780


...

Epoch,Training Loss,Validation Loss,F1
1,1.047400,1.013750,0.533122
2,1.029300,1.039925,0.507284
3,0.883200,1.069589,0.538179


[I 2025-06-16 15:21:30,815] Trial 3 finished with value: 0.5381790776136886 and parameters: {'learning_rate': 2.6014849140916854e-05, 'num_train_epochs': 3, 'per_device_train_batch_size': 16, 'weight_decay': 0.18355605193781274, 'warmup_steps': 740}. Best is trial 1 with value: 0.5545057785535547.


Epoch,Training Loss,Validation Loss,F1
1,1.052900,1.016271,0.542512
2,0.998200,1.017964,0.545909
3,0.925700,1.033108,0.556133
4,0.906200,1.038179,0.549579


[I 2025-06-16 15:26:57,383] Trial 4 finished with value: 0.5495793378953143 and parameters: {'learning_rate': 2.645992710402343e-06, 'num_train_epochs': 4, 'per_device_train_batch_size': 8, 'weight_decay': 0.14569466159380204, 'warmup_steps': 149}. Best is trial 1 with value: 0.5545057785535547.



¡Búsqueda completada!
Mejor trial encontrado:
  -> Valor de la métrica (f1): 0.5545057785535547
  -> Mejores Hiperparámetros: {'learning_rate': 6.998849268339025e-06, 'num_train_epochs': 2, 'per_device_train_batch_size': 32, 'weight_decay': 0.29892488816414, 'warmup_steps': 20}


In [11]:
# --- RE-ENTRENAMIENTO DEL MODELO FINAL CON LOS MEJORES HIPERPARÁMETROS ---

print("\nRe-entrenando el modelo final con los mejores hiperparámetros...")

# 1. Obtén los mejores hiperparámetros del objeto best_run
best_hyperparameters = best_run.hyperparameters

# 2. Crea nuevos TrainingArguments, actualizándolos con los mejores valores
#    Es una buena práctica guardar el modelo final en su propio directorio.
final_training_args = TrainingArguments(
    output_dir="./results_final_model",
    logging_dir="./logs_final_model",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True if torch.cuda.is_available() else False,
    seed=42,
    # ** Actualizamos con los mejores hiperparámetros encontrados **
    learning_rate=best_hyperparameters['learning_rate'],
    num_train_epochs=best_hyperparameters['num_train_epochs'],
    per_device_train_batch_size=best_hyperparameters['per_device_train_batch_size'],
    weight_decay=best_hyperparameters['weight_decay'],
    warmup_steps=best_hyperparameters['warmup_steps'],
)

# 3. Inicializa un nuevo Trainer para el entrenamiento final.
#    Esta vez, pasamos una instancia del modelo directamente con `model`, no `model_init`.
final_trainer = Trainer(
    model=model_init(),  # Llama a la función para obtener un modelo nuevo
    args=final_training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 4. Entrena el modelo final
final_trainer.train()

# 5. ¡Ahora sí! Realiza las predicciones con el trainer final
print("\nRealizando predicciones en el conjunto de evaluación con el modelo final...")
predictions_output = final_trainer.predict(tokenized_eval_dataset)

# --- ANÁLISIS DE RESULTADOS (tu código original, sin cambios) ---

# Las predicciones son logits, convertirlos a etiquetas de clase
predicted_labels = np.argmax(predictions_output.predictions, axis=1)

# Las etiquetas reales (ground truth)
true_labels = predictions_output.label_ids

textos_validacion = df_eval_split['Review'].reset_index(drop=True)

# Creamos el DataFrame
df_resultados = pd.DataFrame({
    'Review': textos_validacion,
    'Etiqueta_Verdadera': true_labels,
    'Prediccion_Modelo': predicted_labels
})



Re-entrenando el modelo final con los mejores hiperparámetros...


/tmp/ipykernel_56794/1780600650.py:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  final_trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1
1,1.060000,1.005725,0.547778
2,0.981000,1.008334,0.554506



Realizando predicciones en el conjunto de evaluación con el modelo final...


In [12]:
# Contar cuántas predicciones fueron correctas
predicciones_correctas = (df_resultados['Etiqueta_Verdadera'] == df_resultados['Prediccion_Modelo']).sum()

# Obtener el número total de predicciones
total_predicciones = len(df_resultados)

# Calcular el porcentaje de acierto
porcentaje_acierto = (predicciones_correctas / total_predicciones) * 100

print(f"\n--- Resultados de Acierto ---")
print(f"Predicciones Correctas: {predicciones_correctas}")
print(f"Total de Muestras de Validación: {total_predicciones}")
print(f"Porcentaje de Acierto (Accuracy): {porcentaje_acierto:.2f}%")


--- Resultados de Acierto ---
Predicciones Correctas: 556
Total de Muestras de Validación: 1000
Porcentaje de Acierto (Accuracy): 55.60%
